PROJECT: H&M Semantic Product Search Engine
DESCRIPTION:
    This script implements a Vector Search (RAG-ready) pipeline to perform 
    semantic search over H&M product descriptions. Instead of keyword matching, 
    it uses deep learning to understand the "meaning" of queries.

WORKFLOW STEPS:
    1. Data Ingestion: Uses DuckDB for high-speed Parquet processing and data cleaning.
    2. Optimization: Deduplicates product descriptions to minimize embedding costs 
       and storage footprint while preserving all associated Article IDs in the metadata.
    3. Embedding: Leverages 'sentence-transformers' (NLP) to convert text into 384-dimensional 
       numerical vectors.
    4. Vector Database: Initializes a Qdrant collection (Docker-based) to store and 
       index these vectors using Cosine Similarity.
    5. Semantic Querying: Transforms natural language queries into vectors to retrieve 
       the most contextually relevant products from the database.

TECH STACK: Python, DuckDB, Qdrant, PyTorch, Sentence-Transformers.

In [ ]:
import pandas as pd
import duckdb
import math
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct, QueryRequest

In [18]:
# ── 1. LOAD & DEDUPLICATE DATA ────────────────────────────────────────────────
# Initialize an in-memory DuckDB connection
con = duckdb.connect()

# Create a virtual view pointing to the local Parquet file (using forward slashes to avoid path errors)
con.execute("CREATE VIEW articles AS SELECT * FROM 'C:/Capstone/parquet/articles.parquet'")

# Extract specific columns while filtering out nulls and generic placeholders
df = con.execute("""
    SELECT article_id, detail_desc
    FROM articles
    WHERE detail_desc IS NOT NULL
      AND detail_desc != 'No description available'
""").df()

# Group by description so we only embed unique text strings once
# This saves memory and processing time by mapping one vector to multiple article IDs
deduped = (
    df.groupby("detail_desc", sort=False)["article_id"]
    .apply(list)
    .reset_index()
    .rename(columns={"article_id": "article_ids"})
)

print(f"Raw rows: {len(df)} → Unique descriptions: {len(deduped)}")


Raw rows: 105126 → Unique descriptions: 43404


In [19]:
# ── 2. GENERATE EMBEDDINGS ────────────────────────────────────────────────────
# Load the pre-trained transformer model (384 dimensions)
model = SentenceTransformer("all-MiniLM-L6-v2")

# Convert the unique descriptions into numerical vectors (embeddings)
embeddings = model.encode(          
    deduped["detail_desc"].tolist(),
    show_progress_bar=True,
    batch_size=64,
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/679 [00:00<?, ?it/s]

In [20]:
# ── 3. SETUP QDRANT COLLECTION ────────────────────────────────────────────────
# Connect to the Qdrant server running in Docker
client = QdrantClient(host="localhost", port=6333, timeout=60)
COLLECTION_NAME = "hm_articles"

# Refresh collection: Delete if it exists to start with a clean slate
if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

# Create a new collection configured for Cosine Similarity
client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)


True

In [22]:
# ── 4. UPLOAD DATA TO QDRANT (UPSERT) ─────────────────────────────────────────
# Prepare the points: each point contains a vector, text, and metadata (payload)
# Instead of one big upsert, do it in chunks
batch_size = 100 
for i in range(0, len(points), batch_size):
    batch = points[i : i + batch_size]
    client.upsert(collection_name=COLLECTION_NAME, points=batch)
    print(f"Uploaded batch {i // batch_size + 1}")

print(f"Successfully uploaded {len(points)} unique vectors to Qdrant.")

Uploaded batch 1
Uploaded batch 2
Uploaded batch 3
Uploaded batch 4
Uploaded batch 5
Uploaded batch 6
Uploaded batch 7
Uploaded batch 8
Uploaded batch 9
Uploaded batch 10
Uploaded batch 11
Uploaded batch 12
Uploaded batch 13
Uploaded batch 14
Uploaded batch 15
Uploaded batch 16
Uploaded batch 17
Uploaded batch 18
Uploaded batch 19
Uploaded batch 20
Uploaded batch 21
Uploaded batch 22
Uploaded batch 23
Uploaded batch 24
Uploaded batch 25
Uploaded batch 26
Uploaded batch 27
Uploaded batch 28
Uploaded batch 29
Uploaded batch 30
Uploaded batch 31
Uploaded batch 32
Uploaded batch 33
Uploaded batch 34
Uploaded batch 35
Uploaded batch 36
Uploaded batch 37
Uploaded batch 38
Uploaded batch 39
Uploaded batch 40
Uploaded batch 41
Uploaded batch 42
Uploaded batch 43
Uploaded batch 44
Uploaded batch 45
Uploaded batch 46
Uploaded batch 47
Uploaded batch 48
Uploaded batch 49
Uploaded batch 50
Uploaded batch 51
Uploaded batch 52
Uploaded batch 53
Uploaded batch 54
Uploaded batch 55
Uploaded batch 56
U

In [24]:
# ── 5. Query — match your query style to the data's style ─────────────────────
# FIX 2: H&M descriptions are short noun phrases ("Floral-print jersey dress").
# Querying with a long natural sentence ("casual summer dress with floral patterns")
# crosses a style gap. Prefix with "product:" if using a model that supports
# asymmetric search, OR just trim the query to match product copy style.

query_text = "pants"
query_vector = model.encode(query_text).tolist()

# FIX: client.search() removed in qdrant-client ≥ 1.10 → use query_points()
results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    limit=5,
).points  # <-- unwrap the ScoredPoint list from the response object

print(f"\nSimilarity search results for: '{query_text}'")
print("-" * 60)
for res in results:
    ids_preview = res.payload["article_ids"][:3]
    more = res.payload["variant_count"] - 3
    id_str = ", ".join(str(i) for i in ids_preview)
    if more > 0:
        id_str += f" (+{more} more)"
    print(f"Score:    {res.score:.4f}")
    print(f"IDs:      {id_str}")
    print(f"Variants: {res.payload['variant_count']}")
    print(f"Text:     {res.payload['raw_text']}")
    print("-" * 60)


Similarity search results for: 'pants'
------------------------------------------------------------
Score:    0.6722
IDs:      0664209001, 0664209002
Variants: 2
Text:     Trousers with a high, elasticated waist and straight, wide legs.
------------------------------------------------------------
Score:    0.6583
IDs:      0660618001, 0660618002, 0660618003 (+3 more)
Variants: 6
Text:     Trousers in stretch twill with a high, elasticated waist and slim legs with a concealed zip at the hems.
------------------------------------------------------------
Score:    0.6563
IDs:      0350135015, 0350135017, 0350135018
Variants: 3
Text:     Trousers in sturdy organic cotton jersey with an elasticated waist and back pockets.
------------------------------------------------------------
Score:    0.6538
IDs:      0557938003
Variants: 1
Text:     5-pocket jeans in washed denim with decorative studs. High waist, low crotch, button fly and straight legs.
-------------------------------------------